# 第 6 章 所得の収束: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/06-income-convergence.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
start_year = 1990  # 成長率を測る期間の始まり（データは 1990〜2019 年）
end_year = 2019    # 期間の終わり
use_controls = False  # True: 初期の資本・人的資本も説明変数に加える

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## β 収斂の回帰

In [ ]:
from lib.chapter02.growth import load_panel
from lib.chapter06.convergence import (
    beta_convergence,
    half_life_years,
    implied_speed,
    income_growth,
)

panel = load_panel()
controls = ("k", "h") if use_controls else ()
result = beta_convergence(panel, start_year, end_year, controls)
beta = float(result.coefficients["ln_y0"])
years = end_year - start_year
print(f"傾き β: {beta:.4f}（初期の所得が高いほど、成長が遅いなら、負）")
print(result.coefficients.round(4).to_string())
print(f"決定係数: {result.r_squared:.3f}")
if beta < 0:
    speed = implied_speed(beta, years)
    print(f"収束の速さ λ: {speed:.4f}、半減期: {half_life_years(speed):.1f} 年")

## 図: 初期の所得と、その後の成長率

In [ ]:
import numpy as np

from lib.chapter03.plot import plot_regression

plot_regression(
    np.log(panel.xs(start_year, level="year")["y"]),
    income_growth(panel, start_year, end_year),
    xlabel=f"Initial income in {start_year} (log)",
    ylabel=f"Growth rate {start_year}-{end_year} (% per year)",
)

## 試してみよう

- `use_controls = True` にすると、初期の所得の係数 β は、−0.4846 から −0.2238 に、
  0 に近づきます。収束の速さから求めた半減期は、
  132.7 年から 299.6 年に伸びます（初期の資本・人的資本を一定にすると、
  収斂は遅く見えます）
- `start_year = 2000` にすると、β は −0.5323 です